In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [36]:
arquivo = r'/home/chlxr/projeto_lean_startup_CD/train_sujo.csv'

In [37]:
df = pd.read_csv(arquivo)

In [38]:
display(df)

,Employee ID,Date of Joining,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
0,fffe32003400330031003000,08-13-2008,male,serv.,S,NaN,3.0,4.3,0.36
1,fffe32003600380032003800,03-11-2008,NaN,SERVICE,YES,2.0,4.0,5.0,0.36
2,fffe32003900330031003700,02-13-2008,male,SERVICE,No,3.0,5.0,7.1,0.68
3,fffe31003100340030003800,10-Aug-08,male,Service,0,4.0,8.0,6.6,0.54
4,fffe32003900320031003800,2008-05-26,F,PRODUCT,n,NaN,7.0,7.5,0.61
...,...,...,...,...,...,...,...,...,...
23883,fffe32003900380034003400,04/04/2008,F,Product,NaN,3.0,NaN,6.8,0.58
23884,fffe32003600360039003200,03-24-2008,Female,Service,N,2.0,4.0,6.6,0.57
23885,fffe3600380039003800,08-30-2008,female,Service,0,1.0,2.0,-5.0,0.40
23886,fffe3700330039003100,10/07/2008,Female,PRODUCT,N,0.0,1.0,4.0,NaN


In [39]:
df.dtypes

Employee ID              object
Date of Joining          object
Gender                   object
Company Type             object
WFH Setup Available      object
Designation             float64
Resource Allocation     float64
Mental Fatigue Score    float64
Burn Rate               float64
dtype: object

In [40]:
print("Contagem total de registros:", df.count())

print("\nNulos por coluna:")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "  Nenhum nulo encontrado.")

Contagem total de registros: Employee ID             23888
Date of Joining         22932
Gender                  22932
Company Type            22932
WFH Setup Available     22932
Designation             22932
Resource Allocation     21535
Mental Fatigue Score    20866
Burn Rate               21807
dtype: int64

Nulos por coluna:
Date of Joining          956
Gender                   956
Company Type             956
WFH Setup Available      956
Designation              956
Resource Allocation     2353
Mental Fatigue Score    3022
Burn Rate               2081
dtype: int64


In [41]:
df.describe()

,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
count,22932.000000,21535.000000,20866.000000,21807.000000
mean,2.178528,4.487532,6.310884,0.452096
std,1.133208,2.041963,7.524564,0.198258
min,0.000000,1.000000,-5.000000,0.000000
25%,1.000000,3.000000,4.500000,0.320000
50%,2.000000,4.000000,5.900000,0.450000
75%,3.000000,6.000000,7.100000,0.590000
max,5.000000,10.000000,99.900000,1.000000


In [42]:
df.columns

Index(['Employee ID', 'Date of Joining', 'Gender', 'Company Type',
       'WFH Setup Available', 'Designation', 'Resource Allocation',
       'Mental Fatigue Score', 'Burn Rate'],
      dtype='object')

In [43]:
df = pd.read_csv(arquivo)

In [44]:
df.dropna(subset=['Burn Rate'], inplace=True)

df['Date of Joining'] = pd.to_datetime(df['Date of Joining'], errors='coerce')
df['Date of Joining'] = df['Date of Joining'].fillna(df['Date of Joining'].median())

df['Designation'] = pd.to_numeric(df['Designation'], errors='coerce')
df['Resource Allocation'] = pd.to_numeric(df['Resource Allocation'], errors='coerce')
df['Mental Fatigue Score'] = pd.to_numeric(df['Mental Fatigue Score'], errors='coerce')

df.loc[~df['Mental Fatigue Score'].between(0, 10), 'Mental Fatigue Score'] = np.nan

cols_texto = ['Gender', 'Company Type', 'WFH Setup Available']

df[cols_texto] = df[cols_texto].apply(
    lambda coluna: coluna.astype(str).str.strip().str.upper()
)

df['Gender'] = df['Gender'].replace({
    'F': 'FEMALE',
    'MULHER': 'FEMALE',
    'FEMININO': 'FEMALE',
    'M': 'MALE',
    'HOMEM': 'MALE',
    'MASCULINO': 'MALE'
})

df['Company Type'] = df['Company Type'].replace({
    'SERV.': 'SERVICE',
    'SERVICO': 'SERVICE',
    'SERVIÇO': 'SERVICE',
    'PROD.': 'PRODUCT',
    'PRODUTO': 'PRODUCT'
})

df['WFH Setup Available'] = df['WFH Setup Available'].replace({
    'Y': 'YES',
    '1': 'YES',
    '1.0': 'YES',
    'SIM': 'YES',
    'S': 'YES',
    'N': 'NO',
    '0': 'NO',
    '0.0': 'NO',
    'NAO': 'NO',
    'NÃO': 'NO'
})

df['Designation'] = df['Designation'].fillna(df['Designation'].mean())
df['Resource Allocation'] = df['Resource Allocation'].fillna(df['Resource Allocation'].mean())
df['Mental Fatigue Score'] = df['Mental Fatigue Score'].fillna(df['Mental Fatigue Score'].mean())

df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Company Type'] = df['Company Type'].fillna(df['Company Type'].mode()[0])
df['WFH Setup Available'] = df['WFH Setup Available'].fillna(df['WFH Setup Available'].mode()[0])

In [45]:
df['WFH Setup Available'] = (
    df['WFH Setup Available']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'YES': 1,
        'Y': 1,
        'SIM': 1,
        'S': 1,
        '1': 1,
        '1.0': 1,

        'NO': 0,
        'N': 0,
        'NAO': 0,
        'NÃO': 0,
        '0': 0,
        '0.0': 0,

        'NAN': np.nan,
        'NONE': np.nan,
        '': np.nan
    })
)

df['WFH Setup Available'] = pd.to_numeric(df['WFH Setup Available'], errors='coerce')
df['WFH Setup Available'] = df['WFH Setup Available'].fillna(df['WFH Setup Available'].mode()[0])


df['Gender'] = (
    df['Gender']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'FEMALE': 0,
        'F': 0,
        'MULHER': 0,
        'FEMININO': 0,

        'MALE': 1,
        'M': 1,
        'HOMEM': 1,
        'MASCULINO': 1,

        'NAN': np.nan,
        'NONE': np.nan,
        '': np.nan
    })
)

df['Gender'] = pd.to_numeric(df['Gender'], errors='coerce')
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])


df['Company Type'] = (
    df['Company Type']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'SERVICE': 0,
        'SERV.': 0,
        'SERVICO': 0,
        'SERVIÇO': 0,

        'PRODUCT': 1,
        'PROD.': 1,
        'PRODUTO': 1,

        'NAN': np.nan,
        'NONE': np.nan,
        '': np.nan
    })
)

df['Company Type'] = pd.to_numeric(df['Company Type'], errors='coerce')
df['Company Type'] = df['Company Type'].fillna(df['Company Type'].mode()[0])

/tmp/ipykernel_8194/1700507220.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({
/tmp/ipykernel_8194/1700507220.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({
/tmp/ipykernel_8194/1700507220.py:62: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({


In [46]:
display(df)

,Employee ID,Date of Joining,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
0,fffe32003400330031003000,2008-08-13,1.0,0.0,1.0,2.178329,3.000000,4.300000,0.36
1,fffe32003600380032003800,2008-03-11,0.0,0.0,1.0,2.000000,4.000000,5.000000,0.36
2,fffe32003900330031003700,2008-02-13,1.0,0.0,0.0,3.000000,5.000000,7.100000,0.68
3,fffe31003100340030003800,2008-07-02,1.0,0.0,0.0,4.000000,8.000000,6.600000,0.54
4,fffe32003900320031003800,2008-07-02,0.0,1.0,0.0,2.178329,7.000000,7.500000,0.61
...,...,...,...,...,...,...,...,...,...
23881,fffe31003600300030003200,2008-07-02,0.0,0.0,0.0,2.178329,4.000000,7.900000,0.58
23883,fffe32003900380034003400,2008-07-02,0.0,1.0,1.0,3.000000,4.491038,6.800000,0.58
23884,fffe32003600360039003200,2008-03-24,0.0,0.0,0.0,2.000000,4.000000,6.600000,0.57
23885,fffe3600380039003800,2008-08-30,0.0,0.0,0.0,1.000000,2.000000,5.728947,0.40


In [47]:
var_indep = ['Gender', 'Company Type',
       'WFH Setup Available', 'Designation', 'Resource Allocation',
       'Mental Fatigue Score']

var_dep = ['Burn Rate']

df_x = df[var_indep]
df_y = df[var_dep]

In [48]:
X_train, X_test, y_train, y_test = train_test_split(df_x, df_y, test_size=0.2, random_state=42) 

print(X_train.shape[0])
print(y_train.shape[0])

17445
17445


In [49]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

linear_model = LinearRegression()
decision_tree = DecisionTreeRegressor(random_state=42)
random_forest = RandomForestRegressor(random_state=42)

In [50]:
linear_model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [51]:
decision_tree.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [52]:
random_forest.fit(X_train, y_train)

/home/chlxr/.local/lib/python3.10/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [53]:
predict_linear = linear_model.predict(X_test)

In [54]:
predict_tree = decision_tree.predict(X_test)

In [55]:
predict_forest = random_forest.predict(X_test)

In [56]:
mae_linear = mean_absolute_error(y_test, predict_linear)
mae_tree = mean_absolute_error(y_test, predict_tree)
mae_forest = mean_absolute_error(y_test, predict_forest)

r_squared_linear = r2_score(y_test, predict_linear)
r_squared_tree = r2_score(y_test, predict_tree)
r_squared_forest = r2_score(y_test, predict_forest)

print("linear: ", mae_linear, r_squared_linear)
print("tree: ", mae_tree, r_squared_tree)
print("forest: ", mae_forest, r_squared_forest)

linear:  0.05919013183459972 0.8404925098221463
tree:  0.05488349593282116 0.8652123688597207
forest:  0.05177207683710656 0.8825060403827846


In [57]:
arquivo_teste = r'/home/chlxr/projeto_lean_startup_CD/test_sujo.csv'
df_teste = pd.read_csv(arquivo_teste)
display(df_teste)

,Employee ID,Date of Joining,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score
0,fffe32003500320038003600,NaN,NaN,PRODUCT,y,3.0,7.0,5.7
1,fffe3200320038003200,02-Apr-08,Homem,Service,N,1.0,2.0,3.3
2,fffe32003400390034003200,2008-06-22,MALE,serv.,YES,0.0,1.0,1.6
3,fffe32003900310036003300,07-27-2008,FeMale,Service,N,2.0,5.0,7.3
4,fffe31003300310033003700,2008-02-27,Female,SERVICE,N,1.0,3.0,6.9
...,...,...,...,...,...,...,...,...
12857,fffe37003000,19/02/2008,m,Service,No,4.0,8.0,8.1
12858,fffe32003200330031003400,2008-07-29,Female,SERVICE,1,2.0,4.0,5.6
12859,fffe33003100390036003400,11-Jan-08,MALE,produto,S,4.0,7.0,6.0
12860,fffe33003200360033003200,05/04/2008,Mulher,Product,N,3.0,5.0,7.9


In [59]:
arquivo_teste = r'/home/chlxr/projeto_lean_startup_CD/test_sujo.csv'
df_teste = pd.read_csv(arquivo_teste)

df_teste['Date of Joining'] = pd.to_datetime(df_teste['Date of Joining'], errors='coerce')
df_teste['Date of Joining'] = df_teste['Date of Joining'].fillna(df_teste['Date of Joining'].median())

df_teste['Designation'] = pd.to_numeric(df_teste['Designation'], errors='coerce')
df_teste['Resource Allocation'] = pd.to_numeric(df_teste['Resource Allocation'], errors='coerce')
df_teste['Mental Fatigue Score'] = pd.to_numeric(df_teste['Mental Fatigue Score'], errors='coerce')

df_teste.loc[~df_teste['Mental Fatigue Score'].between(0, 10), 'Mental Fatigue Score'] = np.nan

cols_texto = ['Gender', 'Company Type', 'WFH Setup Available']

df_teste[cols_texto] = df_teste[cols_texto].apply(
    lambda coluna: coluna.astype(str).str.strip().str.upper()
)

df_teste['Gender'] = df_teste['Gender'].replace({
    'F': 'FEMALE',
    'MULHER': 'FEMALE',
    'FEMININO': 'FEMALE',
    'M': 'MALE',
    'HOMEM': 'MALE',
    'MASCULINO': 'MALE'
})

df_teste['Company Type'] = df_teste['Company Type'].replace({
    'SERV.': 'SERVICE',
    'SERVICO': 'SERVICE',
    'SERVIÇO': 'SERVICE',
    'PROD.': 'PRODUCT',
    'PRODUTO': 'PRODUCT'
})

df_teste['WFH Setup Available'] = df_teste['WFH Setup Available'].replace({
    'Y': 'YES',
    '1': 'YES',
    '1.0': 'YES',
    'SIM': 'YES',
    'S': 'YES',
    'N': 'NO',
    '0': 'NO',
    '0.0': 'NO',
    'NAO': 'NO',
    'NÃO': 'NO'
})

df_teste['Designation'] = df_teste['Designation'].fillna(df_teste['Designation'].mean())
df_teste['Resource Allocation'] = df_teste['Resource Allocation'].fillna(df_teste['Resource Allocation'].mean())
df_teste['Mental Fatigue Score'] = df_teste['Mental Fatigue Score'].fillna(df_teste['Mental Fatigue Score'].mean())

df_teste['Gender'] = df_teste['Gender'].fillna(df_teste['Gender'].mode()[0])
df_teste['Company Type'] = df_teste['Company Type'].fillna(df_teste['Company Type'].mode()[0])
df_teste['WFH Setup Available'] = df_teste['WFH Setup Available'].fillna(df_teste['WFH Setup Available'].mode()[0])

df_teste['WFH Setup Available'] = (
    df_teste['WFH Setup Available']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'YES': 1,
        'Y': 1,
        'SIM': 1,
        'S': 1,
        '1': 1,
        '1.0': 1,

        'NO': 0,
        'N': 0,
        'NAO': 0,
        'NÃO': 0,
        '0': 0,
        '0.0': 0,

        'NAN': np.nan,
        'NONE': np.nan,
        '': np.nan
    })
)

df_teste['WFH Setup Available'] = pd.to_numeric(df_teste['WFH Setup Available'], errors='coerce')
df_teste['WFH Setup Available'] = df_teste['WFH Setup Available'].fillna(df_teste['WFH Setup Available'].mode()[0])


df_teste['Gender'] = (
    df_teste['Gender']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'FEMALE': 0,
        'F': 0,
        'MULHER': 0,
        'FEMININO': 0,

        'MALE': 1,
        'M': 1,
        'HOMEM': 1,
        'MASCULINO': 1,

        'NAN': np.nan,
        'NONE': np.nan,
        '': np.nan
    })
)

df_teste['Gender'] = pd.to_numeric(df_teste['Gender'], errors='coerce')
df_teste['Gender'] = df_teste['Gender'].fillna(df_teste['Gender'].mode()[0])


df_teste['Company Type'] = (
    df_teste['Company Type']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'SERVICE': 0,
        'SERV.': 0,
        'SERVICO': 0,
        'SERVIÇO': 0,

        'PRODUCT': 1,
        'PROD.': 1,
        'PRODUTO': 1,

        'NAN': np.nan,
        'NONE': np.nan,
        '': np.nan
    })
)

df_teste['Company Type'] = pd.to_numeric(df_teste['Company Type'], errors='coerce')
df_teste['Company Type'] = df_teste['Company Type'].fillna(df_teste['Company Type'].mode()[0])

df_teste.to_csv(r'test_limpo.csv', index=False)

ids_funcionarios = df_teste['Employee ID']
X_teste = df_teste[var_indep]

predict_teste = random_forest.predict(X_teste)

def classificar_risco(nota):
    if nota < 0.3:
        return 'Baixo'
    elif nota < 0.6:
        return 'Medio'
    else:
        return 'Alto'

df_result = pd.DataFrame({
    'Employee ID': ids_funcionarios,
    'Burn Rate previsto': predict_teste
})

df_result['Risco'] = df_result['Burn Rate previsto'].apply(classificar_risco)
df_result.to_csv(r'C:\Users\Matheus\Documents\Trabalho Final - Ciência de Dados\predicao.csv', index=False)
display(df_result)

/tmp/ipykernel_8194/1460298025.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_teste['Date of Joining'] = pd.to_datetime(df_teste['Date of Joining'], errors='coerce')
/tmp/ipykernel_8194/1460298025.py:62: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({
/tmp/ipykernel_8194/1460298025.py:92: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({
/tmp/ipykernel_8194/1460298025.p

,Employee ID,Burn Rate previsto,Risco
0,fffe32003500320038003600,0.527874,Medio
1,fffe3200320038003200,0.258300,Baixo
2,fffe32003400390034003200,0.097143,Baixo
3,fffe32003900310036003300,0.588130,Medio
4,fffe31003300310033003700,0.480473,Medio
...,...,...,...
12857,fffe37003000,0.699415,Alto
12858,fffe32003200330031003400,0.424711,Medio
12859,fffe33003100390036003400,0.520022,Medio
12860,fffe33003200360033003200,0.635650,Alto
